In [2]:
import pandas as pd
import os, glob

print("Notebook is running in:", os.getcwd())

# Find each file automatically, skipping the __MACOSX junk folder
def find(filename):
    hits = [p for p in glob.glob(f"**/{filename}", recursive=True) if "__MACOSX" not in p]
    if not hits:
        hits = [p for p in glob.glob(f"../**/{filename}", recursive=True) if "__MACOSX" not in p]
    print(filename, "->", hits)
    return hits[0]

files = {
    "tr_s1": find("train_source1.tsv"),
    "tr_s2": find("train_source2.tsv"),
    "tr_s3": find("train_source3.tsv"),
    "tr_gt": find("train_ground_truth.tsv"),
    "te_s1": find("test_source1.tsv"),
    "te_s2": find("test_source2.tsv"),
    "te_s3": find("test_source3.tsv"),
}

# Load all files (tab-separated, keep everything as text)
data = {}
for name, path in files.items():
    data[name] = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)
    print(name, data[name].shape)

# Show the first 5 rows of each file
for name, df in data.items():
    print("\n=====", name, "=====")
    print(df.head(5).to_string())

# Count records per country in each source file
for name in ["tr_s1", "tr_s2", "tr_s3", "te_s1", "te_s2", "te_s3"]:
    print(name, data[name]["country"].value_counts().to_dict())

Notebook is running in: c:\Users\Rehan\Desktop\AMAZON_ML_CHALLENGE\AMAZON_ML_CHALLENGE
train_source1.tsv -> ['6ab10eb3b23ba_student_resource\\student_resource\\dataset\\train\\train_source1.tsv']
train_source2.tsv -> ['6ab10eb3b23ba_student_resource\\student_resource\\dataset\\train\\train_source2.tsv']
train_source3.tsv -> ['6ab10eb3b23ba_student_resource\\student_resource\\dataset\\train\\train_source3.tsv']
train_ground_truth.tsv -> ['6ab10eb3b23ba_student_resource\\student_resource\\dataset\\train\\train_ground_truth.tsv']
test_source1.tsv -> ['6ab10eb3b23ba_student_resource\\student_resource\\dataset\\test\\test_source1.tsv']
test_source2.tsv -> ['6ab10eb3b23ba_student_resource\\student_resource\\dataset\\test\\test_source2.tsv']
test_source3.tsv -> ['6ab10eb3b23ba_student_resource\\student_resource\\dataset\\test\\test_source3.tsv']
tr_s1 (2206821, 4)
tr_s2 (5034616, 4)
tr_s3 (5285603, 4)
tr_gt (2206821, 2)
te_s1 (1732544, 4)
te_s2 (4887273, 4)
te_s3 (5082316, 4)

===== tr_s1 ===

In [3]:
for name in ["tr_s2", "tr_s3", "tr_gt"]:
    print("\n=====", name, "=====")
    print(data[name].head(8).to_string())

print("\ntr_s1", data["tr_s1"]["country"].value_counts().to_dict())
print("tr_s2", data["tr_s2"]["country"].value_counts().to_dict())

# How many matches does each S1 record have?
gt = data["tr_gt"]
n_matches = gt["matched_entity_ids"].apply(lambda x: 0 if x == "" else len(x.split(",")))
print("\nMatches per S1 record:")
print(n_matches.value_counts().sort_index().head(15))


===== tr_s2 =====
      entity_id                                                             business_name                                                            business_address country
0  S2-166376419                                           राम मार्केटिंग प्राइवेट लिमिटेड                                KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi   India
1  S2-764573417                                              -- Holloway Peak Inc Seafood                                                   105 ELM ST, MORGANTON, NC      US
2  S2-639257739                                                  आदित्य प्रॉपर्टीज एलएलपी                            G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh   India
3  S2-163963287                                                                Summit Inc                                       GREENSBORO, NC, 19 1/2 STARDUST TRAIL      US
4   S2-49942811                                              Delta Tetlecommunication Inc                      

In [4]:
print(data["tr_gt"].head(3).to_dict("records"))
print(n_matches.value_counts().sort_index().to_dict())

[{'source1_entity_id': 'S1-965667', 'matched_entity_ids': 'S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364'}, {'source1_entity_id': 'S1-55344266', 'matched_entity_ids': 'S2-249013014,S2-197070651,S3-478195123,S3-384364074'}, {'source1_entity_id': 'S1-343815751', 'matched_entity_ids': 'S2-790675320,S2-479876582,S3-878454467'}]
{0: 123247, 1: 119157, 2: 375212, 3: 530841, 4: 484115, 5: 321957, 6: 164868, 7: 63968, 8: 18680, 9: 4205, 10: 534, 11: 37}


In [5]:
# Break every ground-truth row into (S1 id, matched id) pairs
gt = data["tr_gt"].copy()
gt = gt[gt["matched_entity_ids"] != ""]
gt["mid"] = gt["matched_entity_ids"].str.split(",")
pairs = gt[["source1_entity_id", "mid"]].explode("mid")
print("Total true pairs:", len(pairs))

# Q1: Does any S2/S3 record belong to more than one S1 record?
counts = pairs["mid"].value_counts()
print("S2/S3 ids matched to MORE than one S1:", (counts > 1).sum())

# Q2: What share of S2 and S3 records are matched to some S1?
matched = set(pairs["mid"])
print("S2 matched share:", round(data["tr_s2"]["entity_id"].isin(matched).mean(), 3))
print("S3 matched share:", round(data["tr_s3"]["entity_id"].isin(matched).mean(), 3))

# Q3: How many S2 and S3 matches does a typical S1 have?
pairs["src"] = pairs["mid"].str[:2]
per = pairs.groupby(["source1_entity_id", "src"]).size().unstack(fill_value=0)
print("S2 matches per S1:", per["S2"].value_counts().sort_index().to_dict())
print("S3 matches per S1:", per["S3"].value_counts().sort_index().to_dict())

Total true pairs: 7638365
S2/S3 ids matched to MORE than one S1: 0
S2 matched share: 0.734
S3 matched share: 0.746
S2 matches per S1: {0: 164498, 1: 789108, 2: 652779, 3: 333957, 4: 119078, 5: 24154}
S3 matches per S1: {0: 143029, 1: 716417, 2: 668375, 3: 372443, 4: 145116, 5: 35378, 6: 2816}


In [8]:
import psutil, os
print("Total RAM (GB):", round(psutil.virtual_memory().total / 1e9, 1))
print("RAM in use now (GB):", round(psutil.Process(os.getpid()).memory_info().rss / 1e9, 1))

os.makedirs("work", exist_ok=True)
for name, df in data.items():
    df.to_pickle(f"work/{name}.pkl")
    print("saved", name)

Total RAM (GB): 16.9
RAM in use now (GB): 3.9
saved tr_s1
saved tr_s2
saved tr_s3
saved tr_gt
saved te_s1
saved te_s2
saved te_s3
